# Phase 0.1: EOS Boundary Validation
## Verify EOS condition: λ_max · η_crit ≈ 2, measure γ vs η relationship

**Protocol:** ConvNet L=5, D=256, BN vs LN, 11 learning rates, 3 seeds, 20 epochs

**Key Measurements:**
- γ (representational shift) vs η (learning rate)
- EOS boundary: λ_max · η ≈ 2

**Data:** CIFAR-10
**Output:** `/kaggle/working/phase0_1_results/`

---
**Kaggle-Adapted:** Checkpoint save/load, SIGTERM graceful exit, BN running stats correctly handled.

In [ ]:
# =============================================================================
# Cell 0: Clone repository from GitHub
# =============================================================================
import os, subprocess, sys

REPO_URL = "https://github.com/xiaonanliu03/thermorg.git"
WORKSPACE = "/kaggle/working/thermorg"
BRANCH = "develop"

if not os.path.exists(WORKSPACE):
    print(f"Cloning {REPO_URL}...")
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, WORKSPACE], check=True)
else:
    print(f"Pulling latest from {BRANCH}...")
    subprocess.run(["git", "pull", "origin", BRANCH], cwd=WORKSPACE, check=True)

sys.path.insert(0, WORKSPACE)
sys.path.insert(0, os.path.join(WORKSPACE, "phase1_experiments"))
print(f"Repository ready: {WORKSPACE}")


In [ ]:
# =============================================================================
# Cell 1: Import from repository
# =============================================================================
import sys
WORKSPACE = "/kaggle/working/thermorg"
sys.path.insert(0, WORKSPACE)

import torch
from phase1_experiments.experiments.train import train_and_measure, run_experiment_grid, CheckpointManager
from phase1_experiments.utils.measurements import measure_gamma, measure_lambda_max_mean, is_stationary, fit_beta
from phase1_experiments.models.convnet import ConvNetL5, create_model
from phase1_experiments.data.datasets import load_cifar10

print(f"Torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")


In [ ]:
# =============================================================================
# Cell 2: Configuration
# =============================================================================
OUTPUT_DIR = '/kaggle/working/phase0_1_results/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

NORM_TYPES = ['batchnorm', 'layernorm']
D = 256
LR_VALUES = [0.0001, 0.0003, 0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 0.5, 1.0, 2.0]
SEEDS = [42, 43, 44]
EPOCHS = 20
BATCH_SIZE = 128

experiment_config = {
    'norm_types': NORM_TYPES,
    'D_values': [D],
    'lr_values': LR_VALUES,
    'seeds': SEEDS,
    'epochs': EPOCHS,
    'checkpoint_every': 5,
}

print(f"Config: {len(NORM_TYPES)} norms, {len(LR_VALUES)} LRs, {len(SEEDS)} seeds, {EPOCHS} epochs")
print(f"Total runs: {len(NORM_TYPES) * len(LR_VALUES) * len(SEEDS)}")


In [ ]:
# =============================================================================
# Cell 3: Load data
# =============================================================================
from phase1_experiments.data.datasets import load_cifar10

trainloader, testloader = load_cifar10(batch_size=BATCH_SIZE, num_workers=0)
print(f"Train batches: {len(trainloader)}, Test batches: {len(testloader)}")


In [ ]:
# =============================================================================
# Cell 4: Run experiment
# =============================================================================
from phase1_experiments.experiments.train import run_experiment_grid

print("Starting experiment grid...")
results = run_experiment_grid(
    experiment_config=experiment_config,
    dataloader_train=trainloader,
    dataloader_eval=testloader,
    device=device,
    output_dir=OUTPUT_DIR,
    verbose=True,
    graceful_exit=None,
)
print(f"Completed {len(results)} runs")


In [ ]:
# =============================================================================
# Cell 5: Analysis
# =============================================================================
import pandas as pd

rows = []
for r in results:
    cfg = r.get('config', {})
    eta = cfg.get('lr', 0) * 128
    rows.append({
        'norm': cfg.get('norm_type'),
        'D': cfg.get('D'),
        'lr': cfg.get('lr'),
        'seed': cfg.get('seed'),
        'gamma': r.get('gamma'),
        'gamma_init': r.get('gamma_init'),
        'lambda_max_init': r.get('lambda_max_init'),
        'eos_ratio': r.get('lambda_max_init', 0) * cfg.get('lr', 0),
        'converged': r.get('is_converged', False),
    })

df = pd.DataFrame(rows)
print(df.groupby('norm').agg({
    'gamma': ['mean', 'std'],
    'lambda_max_init': ['mean', 'std'],
    'eos_ratio': ['mean', 'std']
}).round(4))


In [ ]:
# =============================================================================
# Cell 6: Visualization
# =============================================================================
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: EOS ratio
ax1 = axes[0]
for norm in NORM_TYPES:
    subset = df[df['norm'] == norm]
    ax1.scatter(subset['lr'], subset['eos_ratio'], label=norm, alpha=0.7)
ax1.axhline(2, color='k', linestyle='--', label='EOS target (2)')
ax1.set_xscale('log')
ax1.set_xlabel('Learning Rate')
ax1.set_ylabel('λ_max · η')
ax1.set_title('EOS Boundary Validation')
ax1.legend()

# Right: Gamma vs LR
ax2 = axes[1]
for norm in NORM_TYPES:
    subset = df[df['norm'] == norm]
    ax2.scatter(subset['lr'], subset['gamma'], label=norm, alpha=0.7)
ax2.set_xscale('log')
ax2.set_xlabel('Learning Rate')
ax2.set_ylabel('γ (Representational Shift)')
ax2.set_title('γ vs Learning Rate')
ax2.legend()

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/phase0_1_results.png', dpi=150)
plt.show()
print(f"Saved to {OUTPUT_DIR}/phase0_1_results.png")


In [ ]:
# =============================================================================
# Cell 7: Export results
# =============================================================================
import json
from datetime import datetime

final = {
    'experiment': 'phase0_1_eos_boundary_validation',
    'timestamp': datetime.now().isoformat(),
    'config': experiment_config,
    'results': results,
}

with open(f'{OUTPUT_DIR}/phase_0_1_final_results.json', 'w') as f:
    json.dump(final, f)

print(f"Results saved to {OUTPUT_DIR}/phase_0_1_final_results.json")
